# OLAF-lite — FULL DocRED + FinCausal + EventStoryLine, parallel-5, resumable, budget guarded

This is the full raw-output runner for the same OLAF-lite baseline already smoke-tested.

**Scientific configuration**
- OLAF native `Pipeline`
- POS candidate extraction
- `LLMBasedConceptExtraction`
- `LLMBasedRelationExtraction`
- OpenRouter `openai/gpt-oss-20b`
- reasoning effort `minimal`
- 5 isolated document processes
- gold annotations stripped before OLAF execution

**Cost protections**
- OpenRouter token usage is requested and saved for every LLM call.
- Completed documents are resumed, never paid again.
- The scheduler keeps at most 5 paid documents in flight.
- It stops submitting new work when the tracked budget is reached.
- If token usage is ever missing, it stops submitting new documents after the currently in-flight jobs finish.

This runner generates native OLAF outputs. Benchmark micro-F1 projection/evaluation stays post-hoc.


In [ ]:
from __future__ import annotations

import os, sys, json, shutil
from pathlib import Path
from time import perf_counter
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import multiprocessing as mp

import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
BASE = HERE
while BASE.name != "olaf_baseline" and BASE.parent != BASE:
    BASE = BASE.parent
if BASE.name != "olaf_baseline":
    raise RuntimeError("Run this notebook from inside olaf_baseline.")

SRC = BASE / "src"
VENDOR = BASE / "vendor" / "olaf"
for p in [SRC, VENDOR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv
load_dotenv(BASE / ".env")

from dataset_io import (
    discover_ragtree_preprocessed,
    locate_dataset,
    read_jsonl,
)
from all_datasets_budget_runner import run_one, safe_key, result_path_for

DOCUMENT_WORKERS = 5
MODEL = os.getenv("OLAF_OPENROUTER_MODEL", "openai/gpt-oss-20b")
REASONING_EFFORT = os.getenv("OLAF_REASONING_EFFORT", "minimal")
SPACY_MODEL = "en_core_web_sm"

# OpenRouter published price for openai/gpt-oss-20b on 2026-08-25.
# Change these only if the model page changes.
INPUT_USD_PER_M = 0.03
OUTPUT_USD_PER_M = 0.13

# Hard safety ceiling for TRACKED inference in this full run.
# At most 5 already-in-flight documents can finish after the threshold is crossed.
MAX_TRACKED_USD = 2.00

RUN_ROOT = BASE / "runs" / "olaf_lite_full_3datasets_v1"

assert DOCUMENT_WORKERS == 5
assert "olaf_baseline" in str(sys.executable).lower()
assert ".venv" in str(sys.executable).lower()

print("Python:", sys.executable)
print("Workers:", DOCUMENT_WORKERS)
print("Model:", MODEL)
print("Budget ceiling (tracked inference): $", MAX_TRACKED_USD)
print("Run root:", RUN_ROOT)


In [ ]:
# ZERO-COST preflight and exact dataset selection.
import spacy
spacy.load(SPACY_MODEL)
print("spaCy:", SPACY_MODEL, "OK")

if not os.getenv("OPENROUTER_API_KEY", "").strip():
    raise RuntimeError("OPENROUTER_API_KEY is missing.")

preprocessed = discover_ragtree_preprocessed(BASE)
paths = {
    k: locate_dataset(preprocessed, k)
    for k in ["docred", "fincausal", "eventstoryline"]
}

raw = {k: read_jsonl(p) for k,p in paths.items()}

datasets = {
    "docred": [r for r in raw["docred"] if r.get("type") == "dev"],
    "fincausal": raw["fincausal"],
    "eventstoryline": raw["eventstoryline"],
}

expected = {"docred": 998, "fincausal": 967, "eventstoryline": 443}
actual = {k: len(v) for k,v in datasets.items()}
print("Selected:", actual)
assert actual == expected, (actual, expected)

for k,p in paths.items():
    print(k, "->", p)

print("Total documents:", sum(actual.values()))
print("Expected OLAF LLM calls at ~2/doc:", 2*sum(actual.values()))
print("No API calls made.")


In [ ]:
# Reuse already-paid successful smoke outputs when possible.
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def import_smoke_payload(src: Path, dataset_hint: str | None = None) -> bool:
    try:
        payload = json.loads(src.read_text(encoding="utf-8"))
    except Exception:
        return False

    dataset = str(payload.get("dataset") or dataset_hint or "").lower()
    doc_id = payload.get("document_id")
    if dataset not in datasets or not doc_id:
        return False

    target = result_path_for(RUN_ROOT, dataset, str(doc_id))
    if target.exists():
        return False

    # Make sure this document is actually part of the selected benchmark split.
    valid_ids = {str(r.get("document_id") or r.get("title")) for r in datasets[dataset]}
    if str(doc_id) not in valid_ids:
        return False

    payload = dict(payload)
    payload.setdefault("concept_count", len(payload.get("concepts") or []))
    payload.setdefault("relation_count", len(payload.get("relations") or []))
    payload.setdefault(
        "relations_with_both_endpoints",
        sum(
            1 for r in (payload.get("relations") or [])
            if r.get("source") is not None and r.get("target") is not None
        ),
    )
    payload["prompt_tokens"] = None
    payload["completion_tokens"] = None
    payload["total_tokens"] = None
    payload["llm_calls"] = None
    payload["usage_missing_calls"] = None
    payload["usage_complete"] = False
    payload["estimated_cost_usd"] = None
    payload["resumed"] = True
    payload["imported_from_previous_smoke"] = True

    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return True

imported = 0

parallel_smoke = BASE / "runs" / "docred_parallel5_smoke"
if parallel_smoke.is_dir():
    for p in parallel_smoke.glob("*.json"):
        if not p.name.endswith("_llm_debug.json"):
            imported += int(import_smoke_payload(p, "docred"))

one_each = BASE / "runs" / "smoke_one_each"
if one_each.is_dir():
    for dataset in ["docred", "fincausal", "eventstoryline"]:
        p = one_each / f"{dataset}.json"
        if p.is_file():
            imported += int(import_smoke_payload(p, dataset))

print("Previously-paid smoke documents reused:", imported)
print("No API calls made.")


In [ ]:
# Build pending jobs and summarize existing tracked usage.
jobs = []
for dataset in ["docred", "fincausal", "eventstoryline"]:
    for row in datasets[dataset]:
        doc_id = str(row.get("document_id") or row.get("title"))
        path = result_path_for(RUN_ROOT, dataset, doc_id)
        if not path.is_file():
            jobs.append((dataset, row))

def load_all_results():
    rows = []
    for dataset in ["docred", "fincausal", "eventstoryline"]:
        d = RUN_ROOT / dataset
        if not d.is_dir():
            continue
        for p in d.glob("*.json"):
            if p.name.endswith("_llm_debug.json"):
                continue
            try:
                rows.append(json.loads(p.read_text(encoding="utf-8")))
            except Exception:
                pass
    return rows

existing = load_all_results()
tracked_cost_before = sum(
    float(r.get("estimated_cost_usd") or 0.0)
    for r in existing
    if r.get("estimated_cost_usd") is not None
)

print("Existing completed/imported:", len(existing))
print("Pending:", len(jobs))
print(f"Tracked cost already recorded: ${tracked_cost_before:.6f}")
print(f"Budget remaining: ${MAX_TRACKED_USD - tracked_cost_before:.6f}")
print("No API calls made.")


## Paid full execution

This scheduler continuously keeps up to 5 documents in flight, so one long-tail OpenRouter request does not block the other workers.

The first completed documents immediately give real token/cost telemetry. The notebook prints running spend and a projected full-run cost.


In [ ]:
if tracked_cost_before >= MAX_TRACKED_USD:
    raise RuntimeError(
        f"Tracked budget already reached: ${tracked_cost_before:.6f} >= ${MAX_TRACKED_USD:.2f}"
    )

started = perf_counter()
new_results = []
failures = []
stop_submitting = False
usage_problem = False

job_iter = iter(jobs)
ctx = mp.get_context("spawn")

def submit_next(executor, pending):
    try:
        dataset, row = next(job_iter)
    except StopIteration:
        return False

    fut = executor.submit(
        run_one,
        dataset,
        row,
        str(RUN_ROOT),
        MODEL,
        REASONING_EFFORT,
        SPACY_MODEL,
        INPUT_USD_PER_M,
        OUTPUT_USD_PER_M,
    )
    pending[fut] = (dataset, row)
    return True

with ProcessPoolExecutor(max_workers=DOCUMENT_WORKERS, mp_context=ctx) as executor:
    pending = {}
    for _ in range(DOCUMENT_WORKERS):
        if not submit_next(executor, pending):
            break

    running_cost = tracked_cost_before

    while pending:
        done, _ = wait(set(pending), return_when=FIRST_COMPLETED)

        for fut in done:
            dataset, row = pending.pop(fut)
            doc_id = str(row.get("document_id") or row.get("title"))

            try:
                result = fut.result()
                new_results.append(result)

                cost = result.get("estimated_cost_usd")
                if isinstance(cost, (int, float)):
                    running_cost += float(cost)

                print(
                    f"DONE {dataset:15s} | {doc_id} | "
                    f"{result.get('pipeline_elapsed_seconds', 0):.1f}s | "
                    f"calls={result.get('llm_calls')} | "
                    f"in={result.get('prompt_tokens')} | "
                    f"out={result.get('completion_tokens')} | "
                    f"cost=${float(cost or 0):.6f}"
                )

                if result.get("usage_complete") is False and not result.get("imported_from_previous_smoke"):
                    usage_problem = True
                    stop_submitting = True
                    print("STOP FLAG: OpenRouter usage accounting missing for a newly paid document.")

            except Exception as exc:
                failures.append({
                    "dataset": dataset,
                    "document_id": doc_id,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                })
                print(f"FAILED {dataset} | {doc_id} | {type(exc).__name__}: {exc}")

            # Running budget / projection.
            paid = [
                r for r in new_results
                if isinstance(r.get("estimated_cost_usd"), (int, float))
                and not r.get("resumed", False)
            ]
            if paid:
                avg = sum(float(r["estimated_cost_usd"]) for r in paid) / len(paid)
                currently_complete = len(load_all_results())
                remaining_count = sum(expected.values()) - currently_complete
                projected_remaining = avg * max(0, remaining_count)
                projected_total = running_cost + projected_remaining

                print(
                    f"  spend=${running_cost:.6f} | "
                    f"avg/new-doc=${avg:.6f} | "
                    f"projected total≈${projected_total:.3f}"
                )

            if running_cost >= MAX_TRACKED_USD:
                stop_submitting = True
                print(
                    f"STOP FLAG: tracked budget ${running_cost:.6f} "
                    f"reached ceiling ${MAX_TRACKED_USD:.2f}"
                )

            if not stop_submitting:
                submit_next(executor, pending)

wall = perf_counter() - started
print("\nExecution stopped/finished.")
print("New results this invocation:", len(new_results))
print("Failures:", len(failures))
print("Wall seconds:", round(wall, 2))
print("Usage-accounting stop:", usage_problem)


In [ ]:
# OFFLINE final/current summary.
all_results = load_all_results()
df = pd.DataFrame(all_results)

print("CURRENT COMPLETION")
for dataset,total in expected.items():
    done = sum(1 for r in all_results if r.get("dataset") == dataset)
    print(f"{dataset:15s}: {done}/{total}")
print("TOTAL:", len(all_results), "/", sum(expected.values()))

instrumented = [
    r for r in all_results
    if isinstance(r.get("prompt_tokens"), (int, float))
    and isinstance(r.get("completion_tokens"), (int, float))
]

prompt_tokens = sum(int(r["prompt_tokens"]) for r in instrumented)
completion_tokens = sum(int(r["completion_tokens"]) for r in instrumented)
tracked_cost = sum(float(r.get("estimated_cost_usd") or 0) for r in instrumented)

print("\nTOKEN / COST TELEMETRY")
print("instrumented documents:", len(instrumented))
print("prompt tokens:", prompt_tokens)
print("completion tokens:", completion_tokens)
print("tracked estimated inference cost: $", round(tracked_cost, 6))

if instrumented:
    avg_prompt = prompt_tokens / len(instrumented)
    avg_completion = completion_tokens / len(instrumented)
    avg_cost = tracked_cost / len(instrumented)
    full_docs = sum(expected.values())
    projected_full = avg_cost * full_docs

    print("avg prompt tokens/doc:", round(avg_prompt, 1))
    print("avg completion tokens/doc:", round(avg_completion, 1))
    print("avg cost/doc: $", round(avg_cost, 8))
    print("projected 2408-doc inference cost at observed average: $", round(projected_full, 3))

if not df.empty:
    cols = [
        c for c in [
            "dataset", "document_id", "concept_count", "relation_count",
            "relations_with_both_endpoints", "pipeline_elapsed_seconds",
            "llm_calls", "prompt_tokens", "completion_tokens",
            "estimated_cost_usd", "imported_from_previous_smoke",
        ] if c in df.columns
    ]
    display(df[cols].groupby("dataset").agg(
        documents=("document_id", "count"),
        concepts=("concept_count", "sum"),
        relations=("relation_count", "sum"),
        endpoint_relations=("relations_with_both_endpoints", "sum"),
        mean_pipeline_seconds=("pipeline_elapsed_seconds", "mean"),
    ))

if failures:
    display(pd.DataFrame(failures))

print("\nRun root:", RUN_ROOT)
